In [21]:
import pandas as pd
import joblib

In [22]:
data_path = "dataset/Electronic_sales_Sep2023-Sep2024.csv"
df = pd.read_csv(data_path)

print(df.info())
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  str    
 3   Loyalty Member     20000 non-null  str    
 4   Product Type       20000 non-null  str    
 5   SKU                20000 non-null  str    
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  str    
 8   Payment Method     20000 non-null  str    
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  str    
 13  Shipping Type      20000 non-null  str    
 14  Add-ons Purchased  15132 non-null  str    
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4), str(9)
m

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56
5,1004,41,Female,No,Smartphone,SKU1001,5,Completed,Credit Card,83.00,20.75,4,2024-05-26,Standard,"Impulse Item,Accessory",65.78
6,1005,25,Female,No,Smartwatch,SKU1003,3,Completed,Paypal,7603.47,844.83,9,2024-01-30,Overnight,NaN,0.00
7,1005,25,Female,No,Laptop,SKU1005,3,Completed,Debit Card,4175.64,463.96,9,2024-06-24,Overnight,"Extended Warranty,Extended Warranty",75.33
8,1006,24,Male,No,Smartphone,SKU1004,2,Cancelled,Debit Card,5538.33,791.19,7,2023-10-03,Standard,Impulse Item,43.05
9,1006,24,Male,Yes,Laptop,SKU1005,3,Completed,Cash,4175.64,463.96,9,2024-01-01,Express,NaN,0.00


In [23]:
df = df[df['Order Status'] == 'Completed']
df = df.drop(columns=['Customer ID', 'Age', 'Gender', 'Loyalty Member', 'Rating', 'Payment Method','Order Status', 'Add-ons Purchased', 'Add-on Total', 'Shipping Type'])

df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
df['YearMonth'] = df['Purchase Date'].dt.to_period('M')

df.head()

,Product Type,SKU,Total Price,Unit Price,Quantity,Purchase Date,YearMonth
1,Tablet,SKU1002,741.09,247.03,3,2024-04-20,2024-04
2,Laptop,SKU1005,1855.84,463.96,4,2023-10-17,2023-10
3,Smartphone,SKU1004,3164.76,791.19,4,2024-08-09,2024-08
4,Smartphone,SKU1001,41.50,20.75,2,2024-05-21,2024-05
5,Smartphone,SKU1001,83.00,20.75,4,2024-05-26,2024-05


In [24]:
print("Null Value")
print(df.isnull().sum())

print("\nDuplicated Value")
print(df.duplicated().sum())

Null Value
Product Type     0
SKU              0
Total Price      0
Unit Price       0
Quantity         0
Purchase Date    0
YearMonth        0
dtype: int64

Duplicated Value
2561


In [25]:
df = df.groupby(['YearMonth', 'Product Type', 'SKU']).agg(
    Total_Quantity=('Quantity', 'sum'),
    Total_Revenue=('Total Price', 'sum'),
    Average_Unit_Price=('Unit Price', 'mean')
).reset_index()

df = df.sort_values(by=['SKU', 'YearMonth']).reset_index(drop=True)

print(df.shape[0])
df.head(10)

113


,YearMonth,Product Type,SKU,Total_Quantity,Total_Revenue,Average_Unit_Price
0,2024-01,Headphones,HDP456,1013,365875.34,361.18
1,2024-02,Headphones,HDP456,704,254270.72,361.18
2,2024-03,Headphones,HDP456,895,323256.10,361.18
3,2024-04,Headphones,HDP456,860,310614.80,361.18
4,2024-05,Headphones,HDP456,783,282803.94,361.18
5,2024-06,Headphones,HDP456,958,346010.44,361.18
6,2024-07,Headphones,HDP456,833,300862.94,361.18
7,2024-08,Headphones,HDP456,887,320366.66,361.18
8,2024-09,Headphones,HDP456,627,226459.86,361.18
9,2024-01,Laptop,LTP123,936,631163.52,674.32


In [26]:
df['Year'] = df['YearMonth'].dt.year
df['Month'] = df['YearMonth'].dt.month

df_original = df.copy()

df_model = pd.get_dummies(df, columns=['Product Type', 'SKU'], drop_first=True)

print(df.shape[0])
df_model.head()

113


,YearMonth,Total_Quantity,Total_Revenue,Average_Unit_Price,Year,Month,Product Type_Laptop,Product Type_Smartphone,Product Type_Smartwatch,Product Type_Tablet,SKU_LTP123,SKU_SKU1001,SKU_SKU1002,SKU_SKU1003,SKU_SKU1004,SKU_SKU1005,SKU_SMP234,SKU_SWT567,SKU_TBL345
0,2024-01,1013,365875.34,361.18,2024,1,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2024-02,704,254270.72,361.18,2024,2,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2024-03,895,323256.10,361.18,2024,3,False,False,False,False,False,False,False,False,False,False,False,False,False
3,2024-04,860,310614.80,361.18,2024,4,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2024-05,783,282803.94,361.18,2024,5,False,False,False,False,False,False,False,False,False,False,False,False,False


In [27]:
data_context = {
    'df_original': df_original,
    'df_model': df_model
}
joblib.dump(data_context, 'context/data_context.pkl')

['context/data_context.pkl']